In [1]:
import torch
from dinosaw.wrappers import get_models, ModelTypes
from dinosaw.utils import do_2D_pca, add_custom_font, get_features, hide_axis, apply_labels_as_overlay, COLORS

from interactive_seg_backend import TrainingConfig, FeatureConfig, train_and_apply
from interactive_seg_backend.file_handling import load_labels

/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-07-23 10:26:19 | I | multiscale_classical_cpu.py:  46 | N CPUS: 110
2026-07-23 10:26:19 | W | gpu_utils.py               :  21 | CuPY not installed, GPU fit/apply unavailable!


In [2]:
import numpy as np

from PIL import Image
from skimage.transform import resize
from skimage.color import label2rgb
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from matplotlib.transforms import Bbox
from renderer import SceneLayout, NodeLayout, BoxNode, CubeNode, ImageNode, TextNode, ArrowConnector, render_objects


In [3]:
SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda:0'
half = False

In [4]:
enabled_models: tuple[ModelTypes, ...] = ('dinov2_s', 'alibi_coco_dinov2_s')
models = get_models(enabled_models, DEVICE, True, "../../models/checkpoints")

f_cfg = FeatureConfig()
tr_cfg = TrainingConfig(feature_config=f_cfg, CRF=False, classifier='xgb', CRF_params={"label_confidence": 0.6,
    "sxy_g": [3, 3],
    "sxy_b": [80, 80],
    "s_rgb": [13, 13, 13],
    "compat_g": 10,
    "compat_b": 10,
    "n_infer": 10}, classifier_params={"class_weight": "balanced", "max_depth": 8}) 

2026-07-23 10:26:21 | I | factory.py                 : 152 | Building wrapper 'dinov2_s' on device cuda:0
2026-07-23 10:26:21 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='models', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)


2026-07-23 10:26:22 | I | wrapper.py                 :  48 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12
2026-07-23 10:26:22 | I | factory.py                 : 152 | Building wrapper 'alibi_coco_dinov2_s' on device cuda:0
2026-07-23 10:26:22 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=False, checkpoint_path='../../models/checkpoints/trained/alibi_coco_dv2_vits14_reg_ms.pth', model_conf_path='models', stride=None, remove_pos_embed=True, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[functools.partial(<function add_alibi at 0x7f1df2bd62a0>, slope_type='constant', n_reg_tokens=4, metric='euclidean', normalize=True, wrap=True, add_cls=True, jitter_mag=0.0)], dtype=torch.float32)
2026-07-23 10:26:22 | I | modifications.py           :  47 | Removed default po

In [5]:
sofc_img = Image.open("data/summary/default_image.jpg").convert('RGB')
w, h = sofc_img.size

In [6]:
sofc_feats: dict[ModelTypes, np.ndarray] = {}
for model in enabled_models:
    feats = get_features(models[model], sofc_img)
    feats_red = do_2D_pca(feats, 9, post_norm='minmax')
    feats_red_hr = resize(feats_red, (h, w), order=1)
    sofc_feats[model] = feats_red_hr

2026-07-23 10:26:22 | I | wrapper.py                 :  92 | Processing image, size: [1024, 896]


2026-07-23 10:26:22 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,896,1022] -> f: [1,384,64,73]
2026-07-23 10:26:23 | I | wrapper.py                 :  92 | Processing image, size: [1024, 896]
2026-07-23 10:26:23 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,896,1022] -> f: [1,384,64,73]


In [7]:
preds: dict[ModelTypes, np.ndarray] = {}

for model in enabled_models:
    feats = sofc_feats[model]
    labels = load_labels("data/summary/labels.tiff")[0]
    
    pred, _, _ = train_and_apply(feats, labels, tr_cfg, image=np.array(sofc_img))
    preds[model] = pred

2026-07-23 10:26:23 | I | core.py                    : 155 | Training XGBClassifier: (10692, 9) -> ((10692,)) 
2026-07-23 10:26:23 | I | core.py                    : 170 | Applying XGBClassifier to (896, 1024, 9) features
2026-07-23 10:26:23 | I | core.py                    : 155 | Training XGBClassifier: (10692, 9) -> ((10692,)) 
2026-07-23 10:26:23 | I | core.py                    : 170 | Applying XGBClassifier to (896, 1024, 9) features


/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [10:26:23] WARNING: /workspace/src/learner.cc:793: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [86]:
def render_to_ax(ax) -> None:
    ax = plt.gca()
    ax.set_axis_off()

    R_BOX_WIDTH = 0.25
    R_BOX_HEIGHT = 0.15
    R_BOX_SEP = 0.4


    DINO_COLOR = '#91bbff'
    ALIBI_COLOR = '#EDB458'
    BOX_X = 0.45
    alibi_box = BoxNode(SceneLayout(ax, BOX_X, 0.3, R_BOX_WIDTH, R_BOX_HEIGHT,), fancy=True, facecolor=ALIBI_COLOR)
    alibi_text = TextNode(NodeLayout(alibi_box, (0, 0), 0.1, 0.1, relative_to='center'), r"\textbf{ALiBi-Dv2}", ha="center", va="center", bold=True, fontsize=8)
    remove_pe_text = TextNode(NodeLayout(alibi_text, (0, 0.15), 0.1, 0.1), "(no learned PE)", ha="center", va="center", color="#555555", fontsize=8)

    dino_box = BoxNode(NodeLayout(alibi_box, (0, R_BOX_SEP), R_BOX_WIDTH, R_BOX_HEIGHT,), fancy=True, facecolor=DINO_COLOR)
    dino_text = TextNode(NodeLayout(dino_box, (0, 0), 0.1, 0.1, relative_to='center'), "DINOv2", ha="center", va="center", fontsize=8)
    frozen_text = TextNode(NodeLayout(dino_text, (0, 0.15), 0.1, 0.1), "(frozen)", ha="center", va="center", color="#555555", fontsize=8)

    IMG_L  = 0.2
    img = np.array(Image.open("data/summary/gv/patches.png"))

    IMG_REL_LOC = (-0.35, 0.75 * -IMG_L)
    patch_img = ImageNode(NodeLayout(dino_box, (-BOX_X, 0), IMG_L, IMG_L, "center") , img)


    MAT_L  = IMG_L * 0.95
    DIST_MAT_REL_LOC = (-0.35, -2.5 * MAT_L)
    dist_mat = np.array(Image.open("data/summary/gv/alibi_224.png"))
    dist_mat_img = ImageNode(NodeLayout(patch_img, (0, -0.6), MAT_L, MAT_L), dist_mat)

    CUBE_L = 0.12
    CUBE_DX = 0.3 # 0.325
    dino_cube = CubeNode(NodeLayout(dino_box, (CUBE_DX, 0), CUBE_L,  CUBE_L, "center"), depth_ratio=0.2, color=DINO_COLOR)
    alibi_cube = CubeNode(NodeLayout(alibi_box, (CUBE_DX, 0.0), CUBE_L, CUBE_L, "center"), depth_ratio=0.2, color=ALIBI_COLOR)


    l1 = ArrowConnector(patch_img, dino_box, ('right', 0.5), ('left', 0.5 ), shrinkA=-0.005, shrinkB=0.02)
    l2 = ArrowConnector(l1, None, ('bottom', 0.5), dx=0, dy=-0.4, arrowstyle='-', shrinkA=-0.01, shrinkB=-0.01)
    l3 = ArrowConnector(l2, alibi_box, ('right', 0.0), ('left', 0.5), arrowstyle='->', shrinkA=-0.015, shrinkB=0.02)
    l4 = ArrowConnector(dino_box, dino_cube, ('right', 0.5), ('left', 0.5),  shrinkB=0.01)
    l5 = ArrowConnector(alibi_box, alibi_cube, ('right', 0.5), ('left', 0.5), shrinkB=0.01)
    l6 = ArrowConnector(dino_cube, alibi_cube, ('bottom', 0.5), ('top', 0.5), shrinkA=-0.01, shrinkB=0.03, arrowstyle='<->')


    l7 = ArrowConnector(dist_mat_img, None, ('right', 0.5), dx=0.475, dy=0, arrowstyle='-', shrinkA=-0.005)

    loss_text = TextNode(NodeLayout(l6, (0.05, 0.02), 0.0, 0.0, relative_to='center'), r"\Large $ \mathcal{L}$", fontsize=12, ha="center", va="center", bold=True)
    rel_dist_text = TextNode(NodeLayout(l7, (0.15, -0.14), 0.0, 0.0), "Relative distance matrix", ha="center", va="center", fontsize=8)


    # --- New: Internal and external connectors ---
    # Set N for number of internal/external lines
    N = 5
    internal_lines = []
    external_lines = []
    for i in range(N):
        frac = (i + 0.5) / N  # center them in the box

        ext_start = NodeLayout(l7, (0.45 + frac, 0.05), 0, 0)
        external = ArrowConnector(alibi_box, None, ('bottom', frac), None, dx=0, dy=-0.14,
                                arrowstyle='<-', linestyle='solid')
        external_lines.append(external)


    attn_text = TextNode(NodeLayout(external, (0.15, 0), 0.0, 0.0), "Attention\nlayers", ha="center", va="center", fontsize=8)

    # ---
    lines = [l1, l2, l3, l4, l5, l6, l7]  + external_lines
    boxes = [ alibi_box, dino_box] + internal_lines
    cubes = [dino_cube, alibi_cube]
    images = [patch_img, dist_mat_img]
    text = [alibi_text, frozen_text, dino_text, remove_pe_text, loss_text, rel_dist_text, attn_text]

    objs = lines + boxes + cubes + images + text

    # render_objects(ax, [l1, l2, l3, l4, l5, l6, alibi_box, alibi_text, dino_box, dino_text, patch_img, dist_mat_img, dino_cube, alibi_cube, loss_text], grid_shape=(1, 2))
    render_objects(objs)

In [97]:
W, H = 7, 2.3
fig = plt.figure(figsize=(W, H))

plt.rcParams['text.usetex'] = True
plt.style.use("thesis.mplstyle")
add_custom_font('resources/fonts', 'Grotesk')

add_panel_labels = True      # Add (a), (b), (c) labels

# Width ratios for the 7 logical columns
width_ratios = [1, 1, 1, 1, 1, 1, 1, 1] 

height_ratios = [1, 1]

N_ROWS = len(height_ratios)
N_COLS = len(width_ratios)

fig_w = sum([W * r for r in width_ratios])

fig = plt.figure(figsize=(7, 0.9 * 2.3))
gs = gridspec.GridSpec(
    nrows=N_ROWS,
    ncols=N_COLS,
    width_ratios=width_ratios,
    height_ratios=height_ratios,
    figure=fig,
    wspace=0.3,
    hspace=0.0
)

row_top = 0
row_bottom = 1

# ----------------------------
# Small images (bottom row)
# ----------------------------

axes_small = []

# LHS small images (cols 0,1,2)
for i in range(3):
    ax = fig.add_subplot(gs[row_bottom, i])
    # ax.imshow(small_images[i])
    # ax.set_axis_off()
    hide_axis(ax)
    axes_small.append(ax)

# RHS small images (cols 5,6)
for i, logical_col in enumerate([6, 7]):
    ax = fig.add_subplot(gs[row_bottom, logical_col])
    # ax.imshow(small_images[3 + i])
    # ax.set_axis_off()
    hide_axis(ax)
    axes_small.append(ax)

print(sofc_img, labels)
sofc_with_labels = apply_labels_as_overlay(labels, sofc_img, COLORS, alpha=1)
sofc_img_ax = axes_small[0]
sofc_img_ax.imshow(sofc_with_labels)
sofc_img_ax.set_title("Image\n(+labels)")

dv2_feat_ax = axes_small[1]
dv2_feat_ax.imshow(sofc_feats['dinov2_s'][:, :, :3])
dv2_feat_ax.set_title("Features")

dv2_pred_ax = axes_small[2]
dv2_pred = label2rgb(preds['dinov2_s'] + 1, colors=COLORS[1:], kind='overlay', bg_label=0, image_alpha=1, alpha=1)
dv2_pred_ax.imshow(dv2_pred, interpolation='nearest')
dv2_pred_ax.set_title("Pred")


alibi_feat_ax = axes_small[3]
alibi_feat_ax.imshow(sofc_feats['alibi_coco_dinov2_s'][:, :, :3])
alibi_feat_ax.set_title("Features")

alibi_pred_ax = axes_small[4]
alibi_pred = label2rgb(preds['alibi_coco_dinov2_s'] + 1, colors=COLORS[1:], kind='overlay', bg_label=0, image_alpha=1, alpha=1)
alibi_pred_ax.imshow(alibi_pred, interpolation='nearest')
alibi_pred_ax.set_title("Pred")

# ----------------------------
# Large image (spans two rows, two columns)
# ----------------------------

large_ax = fig.add_subplot(
    gs[row_top:row_bottom + 1, 3:6]
)
cartoon_img = Image.open("data/summary/cartoon.png").convert('RGB')


render_to_ax(large_ax)
large_ax.set_axis_off()

# ----------------------------
# Utility: Draw spanning box
# ----------------------------

def draw_group_box(fig, axes_list, title, bullets, pad=0.01, box_color=None, edge_color='black', bullet_colours=['black', 'black', 'black']):
    # Get union of axes positions in figure coordinates
    bboxes = [ax.get_position() for ax in axes_list]
    union = Bbox.union(bboxes)

    x0, y0 = union.x0, union.y0
    width, height = union.width, union.height

    # Expand vertically to match large graphic height
    large_bbox = large_ax.get_position()
    y0 = large_bbox.y0
    height = large_bbox.height

    rect = patches.FancyBboxPatch(
        (x0 - pad, y0 - pad),
        width + 2 * pad,
        height + 2 * pad,
        boxstyle="round,pad=0.01,rounding_size=0.01",
        transform=fig.transFigure,
        facecolor=box_color if box_color is not None else 'none',
        edgecolor=edge_color,
        linewidth=2,
        zorder=-10
    )
    fig.add_artist(rect)

    # Add title and bullet text
    text_x = x0
    text_y = y0 + height - 0.02

    fig.text(
        text_x,
        text_y,
        title,
        
        fontweight='bold',
        verticalalignment='top'
    )

    for i, bullet in enumerate(bullets):
        fig.text(
            text_x,
            text_y - 0.08 * (i + 1),
            f"{bullet}",
            fontsize=8,
            
            verticalalignment='top',
            color=bullet_colours[i]
        )

# ----------------------------
# Draw LHS and RHS boxes
# ----------------------------

# LHS box: only images 2 and 3 (logical cols 1 and 2)
lhs_axes = axes_small[1:3]
draw_group_box(
    fig,
    lhs_axes,
    title=r"\textbf{DINOv2}",
    bullets=["• Learned PE", "• Positional features", r"$\rightarrow$Biased segmentation"],
    pad=0.0005,
    box_color='#f5e8bc',
    edge_color='#ccc2a1',
    bullet_colours=['black', 'black', 'red']
)
# RHS box: both RHS images
rhs_axes = axes_small[3:5]
draw_group_box(
    fig,
    rhs_axes,
    title=r"\textbf{ALiBi-Dv2}",
    bullets=["• Finetune with ALiBi PE", "• Homogenous features", r"$\rightarrow$Unbiased segmentation"],
    pad=0.0005,
    box_color='#f5e8bc',
    edge_color='#ccc2a1',
    bullet_colours=['black', 'black', 'green']
    # edge_color='#5F978B'
)

# ----------------------------
# Optional panel labels
# ----------------------------

cartoon_bbox = large_ax.get_position()
if add_panel_labels:
    panels = [
        axes_small[0],   # first small (outside box)
        large_ax,        # large graphic
        axes_small[3],   # first RHS small
    ]
    subfig_labels = ["(a)", "(b)", "(c)"]

    for ax, lab in zip(panels, subfig_labels):
        bbox = ax.get_position()
        x = bbox.x0-0.03 if lab =="(c)" else bbox.x0 - 0.02
        fig.text(
            x,
            cartoon_bbox.y1 + 0.05,
            r"\textbf{" + lab + "}",
            
            fontweight=700
        )


ARROW_X = 1.1
ARROW_Y  = -0.25
sofc_img_ax.annotate(
    '', 
    xy=(0.1, ARROW_Y), 
    xycoords='axes fraction', 
    xytext=(3.4, ARROW_Y), 
    arrowprops=dict(
        arrowstyle="<|-",  # larger arrowhead
        color='black', 
        shrinkA=0, 
        shrinkB=0,
    )
)
sofc_img_ax.text(ARROW_X + 0.6, ARROW_Y - 0.325, r'Train classifier (XGB) to map features $\rightarrow$ labels', transform=sofc_img_ax.transAxes,
             rotation=0, va='center', ha='center', fontsize=8)


# plt.show(bbox_inches='tight')

SAVE = True
if SAVE:
    plt.savefig("saved/01.pdf", dpi=300, bbox_inches='tight')
    plt.close()

findfont: Failed to find font weight normal, now using 300.


<PIL.Image.Image image mode=RGB size=1024x896 at 0x7F1D46D524B0> [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


<Figure size 2100x690 with 0 Axes>